# Demographic Population Pyramid Analysis

Population pyramids are mirrored horizontal bar charts that visualize the distribution of age groups and genders within a population. This structural view helps demographers study growth rates and aging trends. This notebook simulates demographic census records, divides them into 10-year age cohorts, and visualizes the structure using an interactive Plotly population pyramid.



In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Generate synthetic census population profiles (100,000 records)
np.random.seed(50)
n_population = 100000

# Generate ages with a realistic demographic distribution (more young people, fewer elderly)
ages = np.random.exponential(scale=38, size=n_population).clip(0, 95)
genders = np.random.choice(['Male', 'Female'], size=n_population, p=[0.495, 0.505])

df_census = pd.DataFrame({
    'Age': ages,
    'Gender': genders
})

# Define 10-year age bins
bins = list(range(0, 101, 10))
labels = [f"{i}-{i+9}" for i in range(0, 90, 10)] + ["90+"]

df_census['Age_Group'] = pd.cut(df_census['Age'], bins=bins, labels=labels, right=False)
df_census.head(10)



,Age,Gender,Age_Group
0,25.931517,Female,20-29
1,9.837379,Female,0-9
2,11.210281,Male,10-19
3,19.179643,Male,10-19
4,18.001157,Male,10-19
5,95.000000,Female,90+
6,19.934109,Female,10-19
7,56.161903,Male,50-59
8,54.315492,Male,50-59
9,14.100935,Male,10-19


## Demographic Aggregations

We calculate total population counts for each combination of age group and gender, and express males as negative values so they plot leftward on the horizontal axis.



In [2]:
# Group and count population
df_counts = df_census.groupby(['Age_Group', 'Gender'], observed=False).size().unstack(fill_value=0)

# Convert to percentage of total population for scaling
total_pop = df_counts.sum().sum()
df_pct = (df_counts / total_pop) * 100

# Make male counts negative for leftward plotting
df_pct['Male_Negative'] = -df_pct['Male']

print("Age Cohort Distribution Percentages (%):")
df_pct



Age Cohort Distribution Percentages (%):


Gender,Female,Male,Male_Negative
Age_Group,,,
0-9,11.577,11.613,-11.613
10-19,9.029,8.817,-8.817
20-29,6.828,6.792,-6.792
30-39,5.271,5.271,-5.271
40-49,4.045,4.117,-4.117
50-59,3.194,3.027,-3.027
60-69,2.374,2.373,-2.373
70-79,1.816,1.890,-1.890
80-89,1.406,1.412,-1.412


## Mirrored Population Pyramid Plot

Using Plotly, we render the population pyramid by overlaying positive female values (right side) and negative male values (left side) along a shared horizontal axis, applying custom styling and formatting.



In [3]:
# Convert structures to standard lists for JSON serialization compatibility
age_groups = list(df_pct.index)
male_pcts = [float(v) for v in df_pct['Male_Negative']]
female_pcts = [float(v) for v in df_pct['Female']]

fig = go.Figure()

# 1. Male Bar Chart Trace (Left Side)
fig.add_trace(go.Bar(
    y=age_groups,
    x=male_pcts,
    orientation='h',
    name='Male',
    marker_color='#3B82F6',
    hoverinfo='text',
    hovertext=[f"Age {grp}: {abs(val):.2f}% Male" for grp, val in zip(age_groups, male_pcts)]
))

# 2. Female Bar Chart Trace (Right Side)
fig.add_trace(go.Bar(
    y=age_groups,
    x=female_pcts,
    orientation='h',
    name='Female',
    marker_color='#EC4899',
    hoverinfo='text',
    hovertext=[f"Age {grp}: {val:.2f}% Female" for grp, val in zip(age_groups, female_pcts)]
))

# Configure horizontal axis styling
fig.update_layout(
    title='Demographic Population Pyramid (% of Total Population)',
    barmode='relative',
    xaxis=dict(
        title='Percentage of Population (%)',
        tickvals=[-6, -4, -2, 0, 2, 4, 6],
        ticktext=['6%', '4%', '2%', '0%', '2%', '4%', '6%']
    ),
    yaxis=dict(title='Age Cohort Group'),
    template='plotly_white',
    width=650,
    height=480,
    legend_title='Gender'
)

fig.show()
